# 01 — Data Acquisition & EDA
**ZivaBasa MVP (Kaggle-Data Phase)**

This notebook:
1. Loads the three proxy Kaggle datasets (Employment, Skills, Productivity task heads)
2. Runs data-quality checks (missing values, duplicates, dtypes)
3. Produces descriptive statistics, correlations, and distribution plots per dataset
4. Documents known cross-dataset alignment limitations (see project README, Section 7)

> ⚠️ These are proxy datasets standing in for real banking-sector data. Findings here are
> methodological (does the pipeline work?), not empirical (are these real workforce dynamics?).

**Output:** lightly-validated raw dataframes saved to `data/raw/` (as parquet) for
`02_feature_engineering.ipynb` to consume.


In [ ]:
# --- Setup ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

print("pandas:", pd.__version__)


## 1. Data Acquisition

Three datasets, one per task head. Download via the Kaggle API (recommended) or manually place
CSVs in `data/raw/` with the filenames below.

```bash
# Kaggle API setup (one-time): place kaggle.json in ~/.kaggle/, then:
pip install kaggle --break-system-packages
kaggle datasets download -d khushikyad001/ai-automation-risk-by-job-role -p ../data/raw --unzip
kaggle datasets download -d pavansubhasht/ibm-hr-analytics-attrition-dataset -p ../data/raw --unzip
kaggle datasets download -d algozee/future-of-work-in-the-age-of-ai-20202026 -p ../data/raw --unzip
```

If you downloaded manually from kaggle.com, just drop the CSVs into `data/raw/` and update the
filenames in the cell below to match.


In [ ]:
# --- Expected raw filenames (adjust to match what you actually downloaded) ---
FILES = {
    "employment": "ai_automation_risk_by_job_role.csv",   # Employment / Automation Risk task head
    "skills": "ibm_hr_attrition.csv",                       # Skills / Readiness task head
    "productivity": "future_of_work_ai_2020_2026.csv",      # Productivity / AI Adoption task head
}

def load_csv(key):
    path = os.path.join(RAW_DIR, FILES[key])
    if not os.path.exists(path):
        print(f"[MISSING] {path} — download it before continuing (see markdown cell above).")
        return None
    df = pd.read_csv(path)
    print(f"[{key}] loaded {df.shape[0]:,} rows x {df.shape[1]} cols from {FILES[key]}")
    return df

df_employment = load_csv("employment")
df_skills = load_csv("skills")
df_productivity = load_csv("productivity")


## 2. Quick Structural Overview

For each dataset: shape, dtypes, first rows. This is the first sanity check before anything else —
confirm you actually got the columns you expect.


In [ ]:
def overview(df, name):
    if df is None:
        print(f"[{name}] not loaded, skipping.")
        return
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print("\nDtypes:\n", df.dtypes)
    display(df.head())
    print("\n" + "-"*80 + "\n")

overview(df_employment, "Employment (automation risk)")
overview(df_skills, "Skills (IBM HR attrition)")
overview(df_productivity, "Productivity (future of work / AI adoption)")


## 3. Data Quality Checks

Missing values, duplicate rows, and constant/near-constant columns — per dataset. Flag issues
here rather than silently handling them; the fixes belong in `02_feature_engineering.ipynb`.


In [ ]:
def quality_report(df, name):
    if df is None:
        return None
    report = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    }).sort_values("missing_pct", ascending=False)

    n_dupes = df.duplicated().sum()
    print(f"=== {name} — Data Quality ===")
    print(f"Duplicate rows: {n_dupes} ({n_dupes / len(df) * 100:.2f}%)")
    display(report)
    print("\n" + "-"*80 + "\n")
    return report

qr_employment = quality_report(df_employment, "Employment")
qr_skills = quality_report(df_skills, "Skills")
qr_productivity = quality_report(df_productivity, "Productivity")


## 4. Descriptive Statistics

Numeric summary stats per dataset — range, mean, std, skew. Useful for spotting outliers and
scale mismatches before feature engineering (e.g. salary in thousands vs. a 1–5 satisfaction score).


In [ ]:
def describe_numeric(df, name):
    if df is None:
        return
    num_df = df.select_dtypes(include=[np.number])
    print(f"=== {name} — Numeric Summary ===")
    display(num_df.describe().T.assign(
        skew=num_df.skew()
    ).round(2))
    print("\n" + "-"*80 + "\n")

describe_numeric(df_employment, "Employment")
describe_numeric(df_skills, "Skills")
describe_numeric(df_productivity, "Productivity")


## 5. Categorical Overview

Value counts for categorical columns — job roles, departments, industries. This tells you what
join keys (if any) might realistically bridge the three datasets, and confirms there's no hidden
target leakage sitting in an unexpected column.


In [ ]:
def categorical_overview(df, name, max_cats=15):
    if df is None:
        return
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    print(f"=== {name} — Categorical Columns ===")
    for col in cat_cols:
        n_unique = df[col].nunique()
        print(f"\n{col} ({n_unique} unique values)")
        if n_unique <= max_cats:
            display(df[col].value_counts())
        else:
            display(df[col].value_counts().head(max_cats))
    print("\n" + "-"*80 + "\n")

categorical_overview(df_employment, "Employment")
categorical_overview(df_skills, "Skills")
categorical_overview(df_productivity, "Productivity")


## 6. Distributions

Histograms for key numeric fields per dataset. Adjust `cols_to_plot` per dataset once you've
confirmed actual column names from Section 2 — these are placeholders based on typical schemas
for each source.


In [ ]:
def plot_distributions(df, cols, name, ncols=3):
    if df is None:
        return
    cols = [c for c in cols if c in df.columns]
    if not cols:
        print(f"[{name}] none of the requested columns were found — check column names.")
        return
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = np.array(axes).reshape(-1)
    for i, col in enumerate(cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i])
        axes[i].set_title(f"{name}: {col}")
    for j in range(len(cols), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

# Adjust these to the real column names once confirmed in Section 2
plot_distributions(df_employment, ["automation_risk", "risk_score", "salary"], "Employment")
plot_distributions(df_skills, ["Age", "MonthlyIncome", "TrainingTimesLastYear", "YearsAtCompany"], "Skills")
plot_distributions(df_productivity, ["ai_adoption_level", "skill_gap_index", "salary_trend"], "Productivity")


## 7. Correlation Analysis

Correlation heatmap per dataset (numeric columns only). Look for: (a) features highly correlated
with your intended target — potential leakage, (b) redundant feature pairs to collapse during
feature engineering.


In [ ]:
def correlation_heatmap(df, name):
    if df is None:
        return
    num_df = df.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        print(f"[{name}] not enough numeric columns for a correlation matrix.")
        return
    plt.figure(figsize=(10, 8))
    sns.heatmap(num_df.corr(), annot=False, cmap="coolwarm", center=0)
    plt.title(f"{name} — Correlation Matrix")
    plt.tight_layout()
    plt.show()

correlation_heatmap(df_employment, "Employment")
correlation_heatmap(df_skills, "Skills")
correlation_heatmap(df_productivity, "Productivity")


## 8. Cross-Dataset Alignment Check

These three datasets are **not** the same population (see README, Section 7 — Known Limitations).
This cell just documents what candidate join/alignment keys exist (role, industry, department
naming) so `02_feature_engineering.ipynb` knows what it's working with — it does **not** attempt
a row-level join.


In [ ]:
candidate_keys = {
    "employment": [c for c in (df_employment.columns if df_employment is not None else [])
                   if any(k in c.lower() for k in ["role", "job", "occupation", "industry"])],
    "skills": [c for c in (df_skills.columns if df_skills is not None else [])
               if any(k in c.lower() for k in ["role", "job", "department", "industry"])],
    "productivity": [c for c in (df_productivity.columns if df_productivity is not None else [])
                      if any(k in c.lower() for k in ["role", "job", "industry", "sector"])],
}

print("Candidate alignment columns per dataset (schema-level only, not a real join key):")
for k, v in candidate_keys.items():
    print(f"  {k}: {v}")

print("""
NOTE: Because these datasets have no shared entity ID, any 'alignment' in
02_feature_engineering.ipynb will be at the feature-schema level (e.g. mapping each dataset's
job-role/industry categories onto a shared taxonomy), not a row-level merge. Document this
explicitly wherever these features are used downstream.
""")


## 9. Save Validated Raw Data

Save each dataframe to parquet (faster + preserves dtypes better than CSV) so
`02_feature_engineering.ipynb` has a clean, consistent starting point. No transformation happens
here — this is a checkpoint, not a cleaning step.


In [ ]:
def save_checkpoint(df, name):
    if df is None:
        print(f"[{name}] not loaded, nothing to save.")
        return
    out_path = os.path.join(RAW_DIR, f"{name}_checked.parquet")
    df.to_parquet(out_path, index=False)
    print(f"[{name}] saved checkpoint -> {out_path}")

save_checkpoint(df_employment, "employment")
save_checkpoint(df_skills, "skills")
save_checkpoint(df_productivity, "productivity")


## 10. Summary — Carry Forward to Notebook 02

Fill this in after running the notebook, before moving to feature engineering:

- [ ] Confirmed actual column names for each dataset (update Sections 6 placeholders if needed)
- [ ] Target variable identified per task head (Employment / Productivity / Skills)
- [ ] Missing-value strategy decided per column (drop / impute / flag)
- [ ] Categorical encoding strategy decided (one-hot vs. target encoding vs. embedding)
- [ ] Outliers noted (from Section 4 skew values) — decide clip vs. keep
- [ ] Cross-dataset alignment approach confirmed (schema-level mapping, not row-level join)
